<div style="
background:#121212;
padding:35px;
border:3px solid #d4af37;
border-radius:15px;
margin:20px 0;
">

<h1 style="
font-size:35px;
color:#d4af37;
text-align:center;
margin-top:0;
margin-bottom:10px;
">
Caso de Negocio 1
</h1>

<h2 style="
font-size:25px;
color:#ffffff;
text-align:center;
font-weight:normal;
margin-top:0;
margin-bottom:25px;
">
Análisis del comportamiento de los saldos bancarios durante 2025
</h2>

<hr style="border:1px solid #d4af37; margin-bottom:30px;">

<h3 style="
font-size:20px;
color:#d4af37;
margin-bottom:10px;
">
Contexto del negocio
</h3>

<p style="
font-size:17px;
color:#e0e0e0;
line-height:1.8;
text-align:justify;
">
Banco Capital GT desea comprender el comportamiento de los saldos de las cuentas de sus clientes durante el año 2025. La gerencia busca identificar patrones temporales, diferencias entre sucursales, segmentos de clientes y productos financieros con el propósito de obtener información que apoye la toma de decisiones comerciales y la gestión de su cartera de clientes.
</p>

<h3 style="
font-size:20px;
color:#d4af37;
margin-top:35px;
margin-bottom:10px;
">
Objetivo general
</h3>

<p style="
font-size:17px;
color:#e0e0e0;
line-height:1.8;
text-align:justify;
">
Analizar el comportamiento de los saldos bancarios registrados durante el año 2025 mediante técnicas de análisis exploratorio de datos para identificar patrones, diferencias entre segmentos de clientes y oportunidades que contribuyan a la toma de decisiones.
</p>

</div>


<div style="
background:#121212;
padding:30px;
border:3px solid #d4af37;
border-radius:15px;
margin:20px 0;
">

<h2 style="
font-size:35px;
color:#d4af37;
text-align:center;
margin-top:0;
margin-bottom:30px;
">
Preguntas de negocio
</h2>

<ol style="
font-size:17px;
color:#e0e0e0;
line-height:2;
padding-left:25px;
">

<li><b>¿Cómo evolucionó el saldo total administrado por el banco durante 2025?</b></li>

<li><b>¿Qué sucursales concentran el mayor saldo de los clientes?</b></li>

<li><b>¿Qué segmentos de clientes concentran los mayores saldos?</b></li>

<li><b>¿Existe una relación entre el ingreso de los clientes y el saldo disponible en sus cuentas?</b></li>

<li><b>¿Qué grupos de edad concentran el mayor saldo administrado por el banco durante 2025?</b></li>

</ol>

</div>

In [7]:
!pip install bokeh
!pip install pandas
!pip install numpy

In [78]:
import pandas as pd

df = pd.read_csv("caso_1_limpios.csv")

df

,cliente_id,fecha,agencia,edad,segmento,ingreso,saldo,producto
0,1,2025-11-01,Quetzaltenango,25,Premium,3819.0,24522.72,3503.06
1,2,2025-11-01,Zona 10,65,Joven,20870.0,35735.08,6087.75
2,3,2025-04-01,Petén,50,Adulto,22726.0,32879.64,5221.01
3,4,2025-08-01,Quetzaltenango,55,Premium,12115.0,25499.88,4787.35
4,5,2025-06-01,Zona 10,35,Joven,8094.0,29405.95,8315.87
...,...,...,...,...,...,...,...,...
995,996,2025-02-01,Escuintla,55,Adulto,27776.0,24017.37,5162.19
996,997,2025-09-01,Quetzaltenango,47,Joven,26273.0,751.93,3162.52
997,998,2025-05-01,Petén,53,Adulto,19075.0,46664.96,5981.30
998,999,2025-09-01,Quetzaltenango,65,Adulto,18186.0,28385.92,8494.76


In [104]:
import numpy as np
import pandas as pd
from bokeh.io import output_notebook
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, RangeTool, HoverTool, NumeralTickFormatter, GlobalInlineStyleSheet, DatetimeTickFormatter, CustomJS
from bokeh.layouts import column
from bokeh.transform import factor_cmap

output_notebook()

THEMES = {
    "luxury_gold_dark": {
        "BACKGROUND": "#0b0b0b",
        "PANEL": "#121212",
        "GRID": "#2c2418",
        "GRID_ALPHA": 0.3,
        "BORDER": "#a57a29",
        "TEXT": "#ffffff",
        "TEXT_SOFT": "#cccccc",
        "ACCENT": "#d4af37",   # dorado
        "POSITIVE": "#b38a2e",
        "TEXT_SIZE": "10pt"
    }
}
theme = THEMES["luxury_gold_dark"]

df_grouped = df.groupby("fecha")["saldo"].sum().reset_index()
dates = np.array(df_grouped["fecha"], dtype=np.datetime64)
source = ColumnDataSource(data=dict(date=dates, saldo=df_grouped["saldo"]))

p = figure(height=300, width=800, tools="xpan,xwheel_zoom,reset,save",
           x_axis_type="datetime", x_axis_location="above",
           background_fill_color=theme["BACKGROUND"],
           border_fill_color=theme["PANEL"],
           x_range=(dates[0], dates[-1]),
           title="Comportamiento de los saldos durante el año 2025"
)

df_grouped["fecha_str"] = pd.to_datetime(df_grouped["fecha"]).dt.strftime("%d-%m-%Y")

source = ColumnDataSource(data=dict(date=pd.to_datetime(df_grouped["fecha"]), 
    fecha_str=df_grouped["fecha_str"], saldo=df_grouped["saldo"]))
hover = HoverTool(
    tooltips=f"""
    <div style="background:{theme['BACKGROUND']}; border:1px solid {theme['ACCENT']}; border-radius:8px; padding:6px 10px; color:{theme['TEXT']};">
        <div><span style="color:{theme['ACCENT']};">Fecha:</span> @fecha_str</div>
        <div><span style="color:{theme['ACCENT']};">Depósitos:</span> @saldo{{0,0}}</div>
    </div>
    """,
    mode="vline"
)

p.add_tools(hover)

p.stylesheets = [
    GlobalInlineStyleSheet(css="""
    .bk-Tooltip {
        --tooltip-color: transparent !important;
        --tooltip-border: transparent !important;
    }
    .bk-tooltip-content {
        background: transparent !important;
        border: none !important;
        box-shadow: none !important;
        padding: 0 !important;
    }
    """)
]

p.yaxis.formatter = NumeralTickFormatter(format="0.00a")
p.toolbar.logo = None
p.toolbar.autohide = True
p.line('date', 'saldo', source=source, line_width=2, color=theme["ACCENT"])
p.title.text_color = theme["ACCENT"] 
p.title.text_font_size = "20pt"        
p.title.align = "center"  
p.yaxis.axis_label = 'Saldos'
p.yaxis.axis_label_text_color = theme["TEXT"]
p.xaxis.major_label_text_color = theme["TEXT_SOFT"]
p.yaxis.major_label_text_color = theme["TEXT_SOFT"]
p.outline_line_color = theme["BORDER"]
p.xgrid.grid_line_color = theme["GRID"]
p.ygrid.grid_line_color = theme["GRID"]
p.xgrid.grid_line_alpha = theme["GRID_ALPHA"]
p.ygrid.grid_line_alpha = theme["GRID_ALPHA"]

select = figure(title="Rango de Temporalidad",
                height=130, width=800,
                x_axis_type="datetime", y_axis_type=None,
                tools="", toolbar_location=None,
                background_fill_color=theme["BACKGROUND"],
                border_fill_color=theme["PANEL"])

range_tool = RangeTool(x_range=p.x_range)
range_tool.overlay.fill_color = theme["ACCENT"]
range_tool.overlay.fill_alpha = 0.3

select.line('date', 'saldo', source=source, color=theme["ACCENT"])
select.ygrid.grid_line_color = None
select.add_tools(range_tool)

show(column(p, select))


Loading BokehJS ...

Evolucion de los depositos

<div style="
background:#121212;
padding:25px;
border-left:6px solid #d4af37;
border-radius:10px;
margin:20px 0;
color:#e0e0e0;        
">

<h3 style="
font-size:28px;
color:#d4af37;
margin-top:0;
">
Análisis
</h3>

<p style="
font-size:18px;
color:#e0e0e0 !important;
line-height:1.8;
text-align:justify;
">

La evolución mensual de los saldos bancarios durante 2025 evidencia un comportamiento caracterizado por fluctuaciones a lo largo del período analizado. El saldo total administrado por el banco oscila entre aproximadamente <b>Q1.79 millones</b> y <b>Q2.55 millones</b>, registrándose el valor más alto en el mes de diciembre, lo que representa una variación cercana a <b>Q800 mil</b> entre el valor mínimo y el máximo observado.

</p>

<p style="
font-size:18px;
color:#e0e0e0;
line-height:1.8;
text-align:justify;
">

No se identifica una tendencia sostenida de crecimiento o disminución durante el año. En su lugar, los saldos presentan variaciones mensuales que podrían estar asociadas a factores estacionales o a cambios en el comportamiento financiero de los clientes.

</p>

<p style="
font-size:18px;
color:#e0e0e0;
line-height:1.8;
text-align:justify;
">

El incremento registrado en diciembre podría estar relacionado con eventos propios de la época, como el pago de aguinaldos, bonos, remesas o un mayor dinamismo de la actividad económica. No obstante, confirmar estas hipótesis requeriría incorporar información adicional, como indicadores macroeconómicos o campañas comerciales desarrolladas por la institución durante el mismo período.

</p>


</div>

In [99]:
df_agencia = (df.groupby("agencia", as_index=False)["saldo"].sum().sort_values("saldo", ascending=True))

source = ColumnDataSource(df_agencia)

p = figure(
    height=450,
    width=850,
    y_range=df_agencia["agencia"],
    toolbar_location="above",
    tools="pan,box_zoom,wheel_zoom,reset,save",
    background_fill_color=theme["BACKGROUND"],
    border_fill_color=theme["PANEL"],
    title="Saldo total por sucursal"
)

p.hbar(
    y="agencia",
    right="saldo",
    height=0.6,
    source=source,
    color=theme["ACCENT"]
)

hover = HoverTool(
    tooltips="""
    <div style="
        background:#0b0b0b;
        border:1px solid #d4af37;
        border-radius:8px;
        padding:8px;
        color:white;
    ">
        <b>Sucursal:</b> @agencia <br>
        <b>Saldo:</b> @saldo{0,0}
    </div>
    """
)

p.add_tools(hover)

p.xaxis.formatter = NumeralTickFormatter(format="0.00a")

p.title.text_color = theme["ACCENT"] 
p.title.text_font_size = "20pt"        
p.title.align = "center"  
p.xaxis.axis_label = "Saldo"
p.yaxis.axis_label = "Sucursal"
p.xaxis.axis_label_text_color = theme["TEXT"]
p.yaxis.axis_label_text_color = theme["TEXT"]
p.xaxis.major_label_text_color = theme["TEXT_SOFT"]
p.yaxis.major_label_text_color = theme["TEXT_SOFT"]
p.outline_line_color = theme["BORDER"]
p.xgrid.grid_line_color = theme["GRID"]
p.ygrid.grid_line_color = theme["GRID"]
p.xgrid.grid_line_alpha = theme["GRID_ALPHA"]
p.ygrid.grid_line_alpha = theme["GRID_ALPHA"]
p.toolbar.logo = None
p.toolbar.autohide = True

show(p)

<div style="
background:#121212;
padding:25px;
border-left:6px solid #d4af37;
border-radius:10px;
margin:20px 0;
color:#e0e0e0;
">

<h3 style="
font-size:28px;
color:#d4af37;
margin-top:0;
">
Análisis
</h3>

<p style="
font-size:18px;
color:#e0e0e0;
line-height:1.8;
text-align:justify;
">

La distribución del saldo total administrado por las agencias presenta diferencias moderadas entre las cinco sucursales analizadas. La agencia de <b>Petén</b> concentra el mayor volumen de saldos, con aproximadamente <b>Q5.34 millones</b>, mientras que la agencia ubicada en <b>Zona 10</b> registra el menor saldo agregado, con alrededor de <b>Q4.71 millones</b>. La diferencia entre ambas asciende a aproximadamente <b>Q500 mil</b>, lo que representa una variación relativamente reducida considerando el volumen total administrado por cada sucursal.

</p>

<p style="
font-size:18px;
color:#e0e0e0;
line-height:1.8;
text-align:justify;
">

En términos generales, no se observa una concentración significativa de los saldos en una única agencia. Los resultados reflejan una distribución relativamente equilibrada entre las sucursales, lo que sugiere que la cartera de clientes y los recursos administrados no dependen de forma predominante de una sola ubicación geográfica.

</p>

<p style="
font-size:18px;
color:#e0e0e0;
line-height:1.8;
text-align:justify;
">

Un aspecto que merece un análisis más detallado es que las agencias ubicadas fuera del área metropolitana presentan niveles de saldos comparables e incluso superiores a la sucursal de Zona 10. Este comportamiento podría estar asociado a diversos factores, como la composición de la cartera de clientes, la presencia de actividades económicas relevantes en dichas regiones, el nivel de bancarización o estrategias comerciales específicas implementadas por la institución. No obstante, la información disponible no permite establecer una relación causal, por lo que sería necesario incorporar variables adicionales para explicar este patrón con mayor precisión.

</p>

</div>

In [100]:
df_segmento = (df.groupby("segmento", as_index=False)["saldo"].sum().sort_values("saldo", ascending=True))

source = ColumnDataSource(df_segmento)

p = figure(
    y_range=list(df_segmento["segmento"]),  
    x_axis_type="linear",
    height=300, width=800,
    tools="xpan,xwheel_zoom,reset,save",
    background_fill_color=theme["BACKGROUND"],
    border_fill_color=theme["PANEL"],
    title="Saldo total por segmento"
)

p.segment(
    x0=0, y0="segmento",
    x1="saldo", y1="segmento",
    source=source,
    line_color=theme["ACCENT"], line_width=5
)


p.scatter(
    x="saldo", y="segmento",
    source=source,
    size=15,
    color=theme["ACCENT"],
    line_color=theme["BORDER"]
)

p.scatter(
    x="saldo", y="segmento",
    source=source,
    size=10,
    color=theme["TEXT"],
    line_color=theme["TEXT"]
)

hover = HoverTool(
    tooltips=f"""
    <div style="
        background:{theme['BACKGROUND']};
        border:1px solid {theme['ACCENT']};
        border-radius:8px;
        padding:6px 10px;
        color:{theme['TEXT']};
    ">
        <div><span style="color:{theme['ACCENT']};">Segmento:</span> @segmento</div>
        <div><span style="color:{theme['ACCENT']};">Depósitos:</span> @saldo{{0.00a}}</div>
    </div>
    """,
    mode="mouse",
    renderers=[p.renderers[-1]]  
)
p.add_tools(hover)

p.xaxis.formatter = NumeralTickFormatter(format="0.00a")

p.title.text_color = theme["ACCENT"] 
p.title.text_font_size = "20pt"        
p.title.align = "center"  
p.xaxis.axis_label_text_color = theme["TEXT"]
p.yaxis.axis_label_text_color = theme["TEXT"]
p.xaxis.major_label_text_color = theme["TEXT_SOFT"]
p.yaxis.major_label_text_color = theme["TEXT_SOFT"]
p.outline_line_color = theme["BORDER"]
p.xgrid.grid_line_color = theme["GRID"]
p.ygrid.grid_line_color = theme["GRID"]
p.xgrid.grid_line_alpha = theme["GRID_ALPHA"]
p.ygrid.grid_line_alpha = theme["GRID_ALPHA"]
p.toolbar.logo = None
p.toolbar.autohide = True

show(p)



<div style="
background:#121212;
padding:25px;
border-left:6px solid #d4af37;
border-radius:10px;
margin:20px 0;
color:#e0e0e0;
">

<h3 style="
font-size:28px;
color:#d4af37;
margin-top:0;
">
Análisis
</h3>

<p style="
font-size:18px;
color:#e0e0e0;
line-height:1.8;
text-align:justify;
">

La distribución del saldo por segmento de clientes muestra una composición relativamente equilibrada entre los tres grupos analizados. El segmento <b>Premium</b> concentra el mayor volumen de saldos, con aproximadamente <b>Q8.58 millones</b>, seguido por el segmento <b>Joven</b> con <b>Q8.34 millones</b> y, finalmente, el segmento <b>Adulto</b>, con alrededor de <b>Q8.01 millones</b>. La diferencia entre el segmento con mayor y menor saldo es cercana a <b>Q500 mil</b>, lo que evidencia una variación moderada entre ellos.

</p>

<p style="
font-size:18px;
color:#e0e0e0;
line-height:1.8;
text-align:justify;
">

En términos generales, no se observa una concentración marcada de los saldos en un único segmento de clientes. Los resultados reflejan una distribución relativamente homogénea, lo que sugiere que el banco mantiene una cartera diversificada y que su volumen de recursos administrados no depende exclusivamente de un perfil específico de clientes.

</p>

<p style="
font-size:18px;
color:#e0e0e0;
line-height:1.8;
text-align:justify;
">

Aunque el segmento <b>Premium</b> registra el mayor saldo agregado, la diferencia respecto a los demás segmentos es reducida. Este comportamiento podría indicar que las estrategias de captación y retención de clientes han logrado una distribución equilibrada de los recursos entre los distintos perfiles. Sin embargo, para comprender las causas de esta distribución sería necesario complementar el análisis con información adicional, como el número de clientes por segmento, el saldo promedio por cliente y la evolución temporal de cada grupo.

</p>
</div>

In [101]:
source = ColumnDataSource(df)

p = figure(
    height=400, width=800,
    tools="pan,wheel_zoom,reset,save,tap",
    x_axis_label="Ingreso (Q)",
    y_axis_label="Saldo disponible (Q)",
    background_fill_color=theme["BACKGROUND"],
    border_fill_color=theme["PANEL"],
    title="Relación saldo - ingreso"
)

renderer = p.scatter(
    x="ingreso", y="saldo",
    source=source,
    size=8,
    fill_color=factor_cmap("segmento", palette=["#d4af37","#29a57a","#377ad4"], factors=df["segmento"].unique()),
    line_color=None
)

hover = HoverTool(tooltips="""
    <div style="background:#0b0b0b; border:1px solid #d4af37; border-radius:6px; padding:6px; color:#fff;">
        <div><b>Cliente:</b> @cliente_id</div>
        <div><b>Ingreso:</b> @ingreso{0,0}</div>
        <div><b>Saldo:</b> @saldo{0,0}</div>
        <div><b>Segmento:</b> @segmento</div>
    </div>
""", renderers=[renderer])
p.add_tools(hover)

callback = CustomJS(args=dict(source=source), code="""
    const indices = source.selected.indices;
    if (indices.length > 0) {
        const seg = source.data['segmento'][indices[0]];
        const new_indices = [];
        for (let i = 0; i < source.data['segmento'].length; i++) {
            if (source.data['segmento'][i] === seg) {
                new_indices.push(i);
            }
        }
        source.selected.indices = new_indices;
    }
""")

taptool = p.select(type=TapTool)
taptool.callback = callback

p.title.text_color = theme["ACCENT"] 
p.title.text_font_size = "20pt"        
p.title.align = "center"  
p.xaxis.axis_label_text_color = theme["TEXT"]
p.yaxis.axis_label_text_color = theme["TEXT"]
p.xaxis.major_label_text_color = theme["TEXT_SOFT"]
p.yaxis.major_label_text_color = theme["TEXT_SOFT"]
p.outline_line_color = theme["BORDER"]
p.xgrid.grid_line_color = theme["GRID"]
p.ygrid.grid_line_color = theme["GRID"]
p.xgrid.grid_line_alpha = theme["GRID_ALPHA"]
p.ygrid.grid_line_alpha = theme["GRID_ALPHA"]

p.toolbar.logo = None
p.toolbar.autohide = True
show(p)


<div style="
background:#121212;
padding:25px;
border-left:6px solid #d4af37;
border-radius:10px;
margin:20px 0;
color:#e0e0e0;
">

<h3 style="
font-size:28px;
color:#d4af37;
margin-top:0;
">
Análisis
</h3>

<p style="
font-size:18px;
color:#e0e0e0;
line-height:1.8;
text-align:justify;
">

El análisis de la relación entre el ingreso de los clientes y el saldo disponible en sus cuentas no evidencia un patrón lineal claramente definido. La dispersión de las observaciones muestra que clientes con niveles de ingreso similares pueden presentar saldos considerablemente diferentes y, de igual forma, clientes con ingresos elevados no necesariamente mantienen los saldos más altos. Este comportamiento se observa tanto en el conjunto de datos como al diferenciar los registros por segmento de clientes.

</p>

<p style="
font-size:18px;
color:#e0e0e0;
line-height:1.8;
text-align:justify;
">

La distribución de los datos sugiere que el nivel de ingreso, por sí solo, no constituye un factor suficiente para explicar el comportamiento de los saldos bancarios. La ausencia de una tendencia claramente definida indica que podrían intervenir otros elementos que no se encuentran representados en la base de datos, tales como los hábitos de ahorro, la antigüedad del cliente, el uso de productos financieros, la actividad económica o características particulares de cada cliente.

</p>

<p style="
font-size:18px;
color:#e0e0e0;
line-height:1.8;
text-align:justify;
">

A partir de este resultado, sería recomendable complementar el análisis mediante técnicas estadísticas, como el cálculo del coeficiente de correlación o la estimación de un modelo de regresión, con el fin de cuantificar la intensidad de la relación entre ambas variables y evaluar si esta resulta estadísticamente significativa. Asimismo, la incorporación de variables adicionales permitiría desarrollar un análisis multivariado que explique con mayor precisión los factores asociados al comportamiento de los saldos bancarios.

</p>

</div>

In [102]:
bins = list(range(0, 105, 5))
labels = [f"{i}-{i+4}" for i in bins[:-1]]

df["grupo_edad"] = pd.cut(df["edad"], bins=bins, labels=labels, right=False)
df_grupo = (df.groupby("grupo_edad", as_index=False)["saldo"].sum().sort_values("grupo_edad"))

source = ColumnDataSource(df_grupo)

p = figure(
    x_range=list(df_grupo["grupo_edad"]),
    height=350, width=750,
    background_fill_color=theme["BACKGROUND"],
    border_fill_color=theme["PANEL"],
    tools="pan,wheel_zoom,reset,save",
    title="Distribución de saldos por edad"
)

p.vbar(
    x="grupo_edad", top="saldo", width=0.75,
    source=source,
    fill_color=theme["ACCENT"], line_color=theme["BORDER"]
)

hover = HoverTool(tooltips="""
    <div style="background:#0b0b0b; border:1px solid #d4af37; border-radius:6px; padding:6px; color:#fff;">
        <div><b>Grupo de edad:</b> @grupo_edad</div>
        <div><b>Saldo total:</b> @saldo{0.00a}</div>
    </div>
""")
p.add_tools(hover)

p.yaxis.formatter = NumeralTickFormatter(format="0.00a")
p.xaxis.major_label_orientation = 1.0  
p.title.text_color = theme["ACCENT"] 
p.title.text_font_size = "20pt"        
p.title.align = "center"  
p.xaxis.axis_label_text_color = theme["TEXT"]
p.yaxis.axis_label_text_color = theme["TEXT"]
p.xaxis.major_label_text_color = theme["TEXT_SOFT"]
p.yaxis.major_label_text_color = theme["TEXT_SOFT"]
p.outline_line_color = theme["BORDER"]
p.xgrid.grid_line_color = theme["GRID"]
p.ygrid.grid_line_color = theme["GRID"]
p.xgrid.grid_line_alpha = theme["GRID_ALPHA"]
p.ygrid.grid_line_alpha = theme["GRID_ALPHA"]

p.toolbar.logo = None
p.toolbar.autohide = True

show(p)

<div style="
background:#121212;
padding:25px;
border-left:6px solid #d4af37;
border-radius:10px;
margin:20px 0;
color:#e0e0e0;
">

<h3 style="
font-size:28px;
color:#d4af37;
margin-top:0;
">
Análisis
</h3>

<p style="
font-size:18px;
color:#e0e0e0;
line-height:1.8;
text-align:justify;
">

La distribución de los saldos por rango de edad durante 2025 muestra que la mayor concentración de recursos se encuentra en los clientes con edades comprendidas entre <b>20 y 69 años</b>. En estos grupos, el saldo agregado oscila entre aproximadamente <b>Q2.15 millones</b> y <b>Q2.70 millones</b>, evidenciando una distribución relativamente homogénea entre las diferentes etapas de la vida laboral.

</p>

<p style="
font-size:18px;
color:#e0e0e0;
line-height:1.8;
text-align:justify;
">

Por el contrario, los grupos ubicados en los extremos de la distribución, correspondientes a clientes de <b>15 a 19 años</b> y de <b>70 a 74 años</b>, registran los menores niveles de saldo, con montos aproximados entre <b>Q464 mil</b> y <b>Q790 mil</b>. La diferencia respecto a los grupos de mayor concentración resulta considerable, lo que sugiere una menor participación de estos rangos de edad en el volumen total de recursos administrados por el banco.

</p>

<p style="
font-size:18px;
color:#e0e0e0;
line-height:1.8;
text-align:justify;
">

Este comportamiento es consistente con el ciclo de vida financiero de los clientes, ya que las edades comprendidas entre los 20 y 69 años suelen concentrar una mayor participación en el mercado laboral y, por consiguiente, una mayor capacidad para generar y mantener saldos en sus cuentas. No obstante, esta interpretación constituye una hipótesis de negocio y requeriría información adicional, como el número de clientes por rango de edad, el saldo promedio por cliente y otras variables socioeconómicas, para ser validada.

</p>




</div>

<div style="
background:#121212;
padding:35px;
border:3px solid #d4af37;
border-radius:15px;
margin:20px 0;
">

<h1 style="
font-size:35px;
color:#d4af37;
text-align:center;
margin-top:0;
margin-bottom:10px;
">
Conclusiones y Recomendaciones
</h1>

<hr style="border:1px solid #d4af37; margin-bottom:30px;">

<h3 style="
font-size:22px;
color:#d4af37;
margin-bottom:10px;
">
Conclusiones
</h3>

<ul style="
font-size:17px;
color:#e0e0e0;
line-height:1.8;
text-align:justify;
">

<li>El saldo total administrado por el banco durante 2025 presentó fluctuaciones mensuales, sin evidenciar una tendencia sostenida de crecimiento o disminución a lo largo del período analizado.</li>

<li>La distribución de los saldos entre las diferentes sucursales fue relativamente equilibrada, sin observarse una concentración significativa en una única agencia. Las diferencias identificadas entre sucursales fueron moderadas en relación con el volumen total administrado.</li>

<li>De forma similar, los segmentos de clientes mostraron una distribución homogénea del saldo agregado, lo que sugiere que la cartera del banco se encuentra diversificada y no depende exclusivamente de un único perfil de clientes.</li>

<li>El análisis de dispersión no evidenció una relación lineal claramente definida entre el ingreso de los clientes y el saldo disponible en sus cuentas, lo que indica que el nivel de ingreso, por sí solo, no explica el comportamiento de los saldos observados.</li>

<li>La mayor concentración de saldos se registró en los clientes con edades comprendidas entre 20 y 69 años, mientras que los grupos de menor y mayor edad presentaron una participación considerablemente menor en el saldo total administrado.</li>

</ul>

<h3 style="
font-size:22px;
color:#d4af37;
margin-top:35px;
margin-bottom:10px;
">
Recomendaciones
</h3>

<ul style="
font-size:17px;
color:#e0e0e0;
line-height:1.8;
text-align:justify;
">

<li>Profundizar el análisis incorporando variables adicionales, como la antigüedad del cliente, el número de productos financieros contratados, la frecuencia de uso de las cuentas y el saldo promedio por cliente, con el fin de identificar los factores que influyen en el comportamiento de los saldos.</li>

<li>Realizar análisis específicos por sucursal y segmento de clientes para identificar oportunidades comerciales, evaluar el desempeño de cada agencia y fortalecer las estrategias de captación y fidelización.</li>

<li>Desarrollar productos y campañas comerciales orientadas a los diferentes segmentos de clientes y grupos de edad, considerando las características y necesidades particulares de cada uno.</li>

<li>Complementar el análisis exploratorio con técnicas estadísticas y modelos predictivos que permitan cuantificar la relación entre las variables analizadas y respaldar futuras decisiones estratégicas basadas en evidencia.</li>

</ul>

</div>

